In [ ]:
%load_ext autoreload
%autoreload 2

import cv2
import dlib
import ipywidgets as widgets
import matplotlib.pyplot as plt
from scipy.spatial import distance 

from driver_stalker.paths import get_data_dpath, get_repo_dpath
from driver_stalker.utils.fs import read_image, get_image_fpaths_from_folder

## Constants

In [ ]:
DATASET_DPATH = get_data_dpath() / "drowsiness_dataset"
assert DATASET_DPATH.exists()

YAWN_DPATH = DATASET_DPATH / "yawn"
assert YAWN_DPATH.exists()

NO_YAWN_DPATH = DATASET_DPATH / "no_yawn"
assert NO_YAWN_DPATH.exists()

In [ ]:
yawn_img_paths = get_image_fpaths_from_folder(YAWN_DPATH)
no_yawn_img_paths = get_image_fpaths_from_folder(NO_YAWN_DPATH)

img_paths = yawn_img_paths + no_yawn_img_paths
len(img_paths)

In [ ]:
face_detector = dlib.get_frontal_face_detector()

face_landmarks_fpath = get_repo_dpath() / "models" / "shape_predictor_81_face_landmarks.dat"
dlib_facelandmark = dlib.shape_predictor(str(face_landmarks_fpath))

In [ ]:
def detect_eye(data):
    point_a = distance.euclidean(data[1], data[5])
    point_b = distance.euclidean(data[2], data[4])
    point_c = distance.euclidean(data[0], data[3])
    return (point_a + point_b) / (2 * point_c)

In [ ]:
@widgets.interact
def demo(index=widgets.IntSlider(value=0, min=0, max=len(img_paths) - 1)):
    img_fpath = img_paths[index]
    img = read_image(img_fpath)

    gray_img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    faces = face_detector(gray_img)

    plt.figure(figsize=(10, 7))

    canva = img.copy()
    for face in faces:
        x0, y0 = face.tl_corner().x, face.tl_corner().y
        x1, y1 = face.br_corner().x, face.br_corner().y
        cv2.rectangle(canva, [x0, y0], [x1, y1], (255, 0, 0), thickness=2)

        face_landmarks = dlib_facelandmark(gray_img, face)

        left_eye_points = [] 
        right_eye_points = [] 

        # right eye
        for n in range(42, 48):
            x = face_landmarks.part(n).x
            y = face_landmarks.part(n).y
            right_eye_points.append((x, y))
            next_point = n+1
            if n == 47:
                next_point = 42
            x2 = face_landmarks.part(next_point).x
            y2 = face_landmarks.part(next_point).y
            cv2.line(canva, (x, y), (x2, y2), (0, 255, 0), 1)
        
        # left eye
        for n in range(36, 42):
            x = face_landmarks.part(n).x
            y = face_landmarks.part(n).y
            left_eye_points.append((x, y))
            next_point = n+1
            if n == 41:
                next_point = 36
            x2 = face_landmarks.part(next_point).x
            y2 = face_landmarks.part(next_point).y
            cv2.line(canva, (x, y), (x2, y2), (255, 255, 0), 1)

        right_eye_ratio = detect_eye(right_eye_points)
        left_eye_ratio = detect_eye(left_eye_points)
        
        eye_ratio = (left_eye_ratio+right_eye_ratio)/2
        eye_ratio = round(eye_ratio, 2)

        if eye_ratio < 0.25:
            plt.title("DROWSINESS DETECTED! ALERT!")

    plt.imshow(canva)
    plt.show()